### Imports


In [1]:
import os
from dotenv import load_dotenv
import json
from openai import OpenAI
import gradio as gr
import sqlite3
import pandas as pd
import numpy as np


### Creating Environment and Model

In [2]:
load_dotenv()

api_key = os.getenv("GEMINI_API_KEY", "")

if not api_key:
    raise ValueError("GEMINI_API_KEY environment variable not set")
else:
    print("Api key loaded successfully and stars with:", api_key[:4])
    
BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai"
MODEL = "gemini-2.5-flash-lite"

gemini = OpenAI(base_url=BASE_URL, api_key=api_key)

Api key loaded successfully and stars with: AIza


### Create exercise database

**Load exercises data from csv**

In [115]:
df = pd.read_csv("megaGymDataset.csv", index_col="Unnamed: 0")
df.drop(columns=["Rating", "RatingDesc", "Type"], inplace=True)
df.rename(columns={"Title": "name", "BodyPart": "muscle", "Desc": "description", "Equipment": "equipment", "Level": "level"}, inplace=True)
df["muscle"]= df["muscle"].str.lower()

level_map= {
    "Beginner": "1",
    "Intermediate": "2",
    "Expert": "3"
}
df["level"] = df["level"].map(level_map).astype(int)
print(df.head())

                           name  \
0        Partner plank band row   
1  Banded crunch isometric hold   
2         FYR Banded Plank Jack   
3                 Banded crunch   
4                        Crunch   

                                         description      muscle equipment  \
0  The partner plank band row is an abdominal exe...  abdominals     Bands   
1  The banded crunch isometric hold is an exercis...  abdominals     Bands   
2  The banded plank jack is a variation on the pl...  abdominals     Bands   
3  The banded crunch is an exercise targeting the...  abdominals     Bands   
4  The crunch is a popular core exercise targetin...  abdominals     Bands   

   level  
0      2  
1      2  
2      2  
3      2  
4      2  


In [66]:
print(df["muscle"].unique())

['abdominals' 'adductors' 'abductors' 'biceps' 'calves' 'chest' 'forearms'
 'glutes' 'hamstrings' 'lats' 'lower back' 'middle back' 'traps' 'neck'
 'quadriceps' 'shoulders' 'triceps']


In [67]:
print(df["equipment"].unique())

['Bands' 'Barbell' 'Kettlebells' 'Dumbbell' 'Other' 'Cable' 'Machine'
 'Body Only' 'Medicine Ball' nan 'Exercise Ball' 'Foam Roll'
 'E-Z Curl Bar']


In [68]:
print(df["level"].unique())

['2' '1' '3']


**Create database with exercises**

In [5]:
DB = "exercises.db"
with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute('CREATE TABLE IF NOT EXISTS exercises (name TEXT PRIMARY KEY, muscle TEXT, description TEXT, equipment TEXT, level INTEGER)')
    conn.commit()
    
    

In [117]:
with sqlite3.connect(DB) as conn:
    df.to_sql("exercises", conn, if_exists="replace")

### Create Tool

**Exercise retrieval tool**

In [6]:
def get_exercise(muscle: str, level: int):
    print(f"DATABASE TOOL CALLED: Getting exercise for target muscle {muscle} and level {level}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT name FROM exercises WHERE muscle LIKE ? AND level <= ?', (f'%{muscle}%', level))        
        result = cursor.fetchall()
        if result:
            return result  # Return a random exercise name from the results
        else:
            return "No exercise found for the specified target muscle and level."

In [7]:
get_exercise("chest", 3)

DATABASE TOOL CALLED: Getting exercise for target muscle chest and level 3


[('Cross Over - With Bands',),
 ('Bench Press - With Bands',),
 ('Bench Press With Short Bands',),
 ('Feet-Elevated TRX Push-Up',),
 ('Band-suspended kettlebell bench press',),
 ('HM Banded Cross-Over Pull',),
 ('Rusin Band Pull-Apart',),
 ('UNS Banded Push-Up',),
 ('Taylor Band-Assisted One-Arm Push-Up',),
 ('Incline band bench press',),
 ('Band push-up',),
 ('Band chest fly',),
 ('Close-grip bench press',),
 ('Barbell Bench Press - Medium Grip',),
 ('Decline barbell bench press',),
 ('Wide-grip bench press',),
 ('Wide-Grip Decline Barbell Bench Press',),
 ('Barbell Incline Bench Press Medium-Grip',),
 ('Neck Press',),
 ('Front Raise And Pullover',),
 ('Board bench press',),
 ('Barbell Bench Press-Wide Grip',),
 ('Wide-Grip Decline Barbell Pullover',),
 ('Barbell Guillotine Bench Press',),
 ('Band-suspended kettlebell bench press',),
 ('Paul Carter Incline Bench Press',),
 ('Incline bench press',),
 ('Bench press',),
 ('TBS Close-Grip Bench Press',),
 ('AM Flat Bench Barbell Press',),

In [8]:
exercise_function = {
    "name": "get_exercise",
    "description": "Get an exercise for a specific muscle group and difficulty level.",
    "parameters": {
        "type": "object",
        "properties": {
            "muscle": {
                "type": "string",
                "description": "The muscle group that the exercise targets, e.g., 'biceps', 'triceps', 'legs', etc.",
            },
            "level": {
                "type": "integer",
                "description": "The difficulty level of the exercise, e.g., 1 for Beginner, 2 for Intermediate, 3 for Expert.",
            },
        },
        "required": ["muscle", "level"],
        "additionalProperties": False
    }
}

In [9]:
tools = [{"type": "function", "function": exercise_function}]

**Call tool function**

In [10]:
tools_map = {
    "get_exercise": get_exercise
}

In [11]:
def handle_tool_calls(message):
    response = []
    for tool_call in message.tool_calls:
        print(f"Tool call: {tool_call.function.name} with arguments {tool_call.function.arguments}", flush=True)
        function_name = tool_call.function.name
        if function_name in tools_map:
            function = tools_map[function_name]
            arguments = json.loads(tool_call.function.arguments)
            exercises = function(**arguments)
            response.append({
                "role": "tool",
                "content": json.dumps(exercises),
                "tool_call_id": tool_call.id
            })
    return response

### Create coach

In [28]:
system_prompt = """
    You are a professional online coach. Your task is to help gym members achieve their fitness goals by providing personalized
    workout plans and motivation. You will ask questions to understand the user's fitness level, goals,
    and preferences before creating a tailored plan. Always encourage and support the user in their fitness journey.
    
    You should start by asking the user about their fitness level. 
    Then follow up with their fintess goals. 
    Ask how much time they can dedicate to working out each week. 
    Finally, ask about any preferences or limitations they may have (e.g., equipment, injuries).
    
    If a user tells you these details on their own without you asking, you can skip asking about them and just use the information they provided to create the workout plan.
    Their fitness level and days of training per week are the most important details you need to create a workout plan, so if they provide those on their own, 
    you can skip asking about them and just use that information to create the workout plan using the get_exercise tool.
    
    If a person only ask for some exercises for a certain muscle group, you can skip the workout plan overview and just recommend exercises
    for that muscle group by retrieving them from the database using the get_exercise tool.
    
    If a user is beginner (level 1), don't recommend training more than 3 times a week and focus on full-body workouts or upper/lower splits. Do not 
    recommend using heavy weights or advanced exercises. Focus on bodyweight exercises, machines, and light dumbbells. Do not recommend more 
    than 2 exercises per muscle group per workout since they are training full body. Also don't recommend doing more than 6-8 exercises per
    workout. Either do 1 exercise per muscle group with 3-4 sets or 2 exercises per muscle group with 2-3 sets.
    
    If they are intermediate (level 2), recommend training 3-5 times a week, but not more than 3 days in a row. Focus on a mix of full-body workouts,
    upper/lower splits, push/pull/legs, and any similar splits. You can recommend using moderate weights and a mix of machines, dumbbells,
    and bodyweight exercises. Do not recommend training 7 days a week. Do not recommend doing more than 6-8 exercises per workout and 2-3
    exercises per muscle group. When doing 2 exercises per muscle group, recommend doing 3-4 sets for the first exercise and 2-3 sets for
    the second exercise. When doing 3 exercises per muscle group, recommend doing 2-3 sets for each exercise.
    
    If they are advanced (level 3), recommend training 5-6 times a week, but not more than 3 days in a row. You should not recommend training 7 days
    a week. If they want to train 6 days a week, recommend doing an 8 day split with one rest day in the middle. You should not recommend 
    they full body split any more, nor training only one muscle group per day. Focus on upper/lower splits, push/pull/legs, and similar splits.
    Recommend using heavier weights and more advanced exercises with a focus on progressive overload and periodization. Lower set and rep
    ranges for advanced lifters, with an emphasis on strength and hypertrophy training. 2-3 exercises per muscle group and 2, but not more
    than 3 sets per exercise.
    
    Before recommending the exercises give the user a workout plan overview with the splits and muscle groups they will train each day.
    Then for each day, after the user has confirmed the workout plan, recommend exercises for each muscle group.
    
    When creating a workout plan, you should always use the get_exercise tool, to find exercises that target specific muscle groups and
    are appropriate for the user's fitness level. Do not select one exercise more than once per workout. Try to choose not redundant
    exercises that target the same muscle group in a similar way. Don't recommend any exercise that is not in the database. This is the list of muscle groups
    you can search exercises for: ['abdominals' 'adductors' 'abductors' 'biceps' 'calves' 'chest' 'forearms' 'glutes' 'hamstrings' 'lats' 'lower back' 'middle back' 'traps' 'neck'
    'quadriceps' 'shoulders' 'triceps']. So for example, if you want to recommend exercises for legs, you can search for exercises for quadriceps, hamstrings, calves, abductors,
    and adductors, but you should not search for exercises for legs since that is not a muscle group in the database.
    
    CRITICAL: You are FORBIDDEN from naming any exercise from your own memory. You MUST call the get_exercise tool to retrieve exercises from the official database before providing them to the user.
    
    Keep you questions and responses concise and to the point. Only explain exercises to beginners and if the user asks for clarification.
    Always end your response with a question to keep the conversation going.
    
    When aksed about anything that is not related to exercise recommendations, workout plans, or fitness advice, tell the user that you are an online 
    fitness coach and can only provide advice related to fitness and workout plans.
"""

In [29]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = gemini.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    response_message = response.choices[0].message

    if response_message.tool_calls:
        messages.append(response_message)
        
        for tool_call in response_message.tool_calls:
            print(f"Tool call: {tool_call.function.name} with arguments {tool_call.function.arguments}", flush=True)
            tool_response = handle_tool_calls(response_message)
            messages.append(tool_response)
        
        # Second call to get the final text summary
        final_response = gemini.chat.completions.create(model=MODEL, messages=messages)
        return final_response.choices[0].message.content # Always return the .content string
    
    return response_message.content

### Launch Interface

In [31]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7869
* To create a public link, set `share=True` in `launch()`.
